# 02 — Custom MLP Head Training on Perch Embeddings

Trains a compact MLP head (1536 → 512 → 512 → 234) on the embeddings from notebook `01`, using a per-class-thresholded pseudo-labelling scheme over the unlabelled soundscapes plus the 708 labelled windows. This is the candidate ensemble component evaluated in notebook `03`.

**Author:** Edidiong-Abasi Anwanane. **Own work.**

**Outputs (not committed):** trained head weights — reserved for follow-up work.

Run on Kaggle (T4), ~30–60 min.

In [ ]:
# =============================================================================
# BirdCLEF+ 2026 — Path Y Day 2: Train MLP head on Perch v2 embeddings
# =============================================================================
# What this does:
#   1. Load the 708 real-labeled embeddings (perch_meta dataset)
#   2. Load the 127,896 unlabeled embeddings (your day1p5 dataset)
#   3. Generate pseudo-labels by thresholding Perch logits at logit > 2.0
#   4. Train a small MLP (1536 -> 512 -> 234) with GroupKFold-by-filename
#   5. Save best checkpoint as head_v01.pth
#
# Output: /kaggle/working/head_v01.pth (~10-20 MB)
#         /kaggle/working/train_log.json
#         /kaggle/working/per_class_auc.csv
#
# Setup BEFORE running:
#   • Accelerator: GPU T4 x2 (we use one)
#   • Internet: ON (just in case)
#   • Inputs attached:
#       - BirdCLEF+ 2026 (competition data)
#       - jaejohn/perch-meta
#       - didianwanane/birdclef-perch-soundscapes-embs-v01
#
# Recommended: use "Save & Run All (Commit)" for the actual run, not
# interactive. That way the output checkpoint is properly captured
# in the committed version's Output tab and you can create the Day 4
# checkpoint dataset directly without local download/upload.
#
# Estimated runtime: 30-60 minutes on T4 GPU.
# =============================================================================

# =============================================================================
# CELL 1: Verify environment
# =============================================================================
import importlib
needed = ["numpy", "pandas", "torch", "sklearn", "scipy"]
for pkg in needed:
    try:
        m = importlib.import_module(pkg)
        print(f"✓ {pkg} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"✗ {pkg}: {e}")

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# =============================================================================
# CELL 2: Imports & config
# =============================================================================
import os, sys, re, gc, time, json, math, random, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")


class CFG:
    # Paths
    COMP_DIR        = "/kaggle/input/competitions/birdclef-2026"
    PERCH_META      = "/kaggle/input/datasets/jaejohn/perch-meta"    
    EMBED_DATASET   = "/kaggle/input/datasets/didianwanane/birdclef-perch-soundscapes-embs-v01"
    OUT_DIR         = "/kaggle/working"

    # Model
    EMB_DIM         = 1536      # Perch v2 embedding dimension
    HIDDEN_DIM      = 512
    N_CLASSES       = 234

    # Training
    BATCH_SIZE      = 1024
    EPOCHS          = 25
    LR              = 1e-3
    WEIGHT_DECAY    = 1e-4
    DROPOUT         = 0.3
    LABEL_SMOOTH    = 0.0
    GRAD_CLIP       = 5.0
    NUM_WORKERS     = 2
    SEED            = 42

    # Pseudo-labels
    PSEUDO_LOGIT_THRESHOLD = 2.0   # Perch logit above which to treat as positive pseudo-label
    PSEUDO_SAMPLE_WEIGHT   = 0.3   # Pseudo-labeled samples weighted lower than real labels
    REAL_SAMPLE_WEIGHT     = 1.0
    MIN_POSITIVES_PER_PSEUDO = 1   # Require at least 1 positive logit above threshold

    # Validation
    N_FOLDS         = 5
    FOLD_TO_TRAIN   = 0
    SANITY_AUC_FLOOR = 0.65        # Abort if epoch 0 val_auc below this


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_all(CFG.SEED)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
os.makedirs(CFG.OUT_DIR, exist_ok=True)

In [ ]:
# =============================================================================
# CELL 3: Build class space + sanity-check paths
# =============================================================================
sample_sub = pd.read_csv(f"{CFG.COMP_DIR}/sample_submission.csv")
SPECIES = [c for c in sample_sub.columns if c != "row_id"]
assert len(SPECIES) == CFG.N_CLASSES, f"Expected {CFG.N_CLASSES} species, got {len(SPECIES)}"
SPECIES_TO_IDX = {s: i for i, s in enumerate(SPECIES)}
print(f"✓ {len(SPECIES)} species in class space")

# Verify input datasets exist
assert os.path.exists(CFG.PERCH_META), f"perch-meta not at {CFG.PERCH_META}"
assert os.path.exists(CFG.EMBED_DATASET), f"embed dataset not at {CFG.EMBED_DATASET}"
print(f"✓ perch-meta at {CFG.PERCH_META}")
print(f"✓ embed dataset at {CFG.EMBED_DATASET}")

In [ ]:
# =============================================================================
# CELL 4: Load labeled data (perch_meta — 708 windows with real labels)
# =============================================================================
print("=" * 70)
print("Loading labeled data from perch_meta")
print("=" * 70)

pm_npz = np.load(f"{CFG.PERCH_META}/full_perch_arrays.npz")
pm_meta = pd.read_parquet(f"{CFG.PERCH_META}/full_perch_meta.parquet")
pm_embs   = pm_npz["emb_full"]            # (708, 1536)
pm_scores = pm_npz["scores_full_raw"]     # (708, 234) - Perch logits, not what we want as labels
print(f"perch_meta embs:    {pm_embs.shape}")
print(f"perch_meta scores:  {pm_scores.shape}")
print(f"perch_meta meta:    {pm_meta.shape}")

# Build REAL labels from train_soundscapes_labels.csv (the human-annotated truth)
ss_labels = pd.read_csv(f"{CFG.COMP_DIR}/train_soundscapes_labels.csv")
def time_to_sec(t):
    parts = str(t).split(":")
    return int(float(parts[0])*3600 + float(parts[1])*60 + float(parts[2]))

ss_labels["end_sec"] = ss_labels["end"].apply(time_to_sec)
ss_labels["row_id"] = ss_labels["filename"].str.replace(".ogg", "", regex=False) + "_" + ss_labels["end_sec"].astype(str)

def parse_ss(s):
    if pd.isna(s) or s == "":
        return []
    return [x.strip() for x in str(s).split(";") if x.strip()]

# Group by row_id (some row_ids may have multiple label rows)
row_to_labels = defaultdict(set)
for _, r in ss_labels.iterrows():
    for lbl in parse_ss(r["primary_label"]):
        if lbl in SPECIES_TO_IDX:
            row_to_labels[r["row_id"]].add(SPECIES_TO_IDX[lbl])

# Build y_real array aligned with pm_meta rows
y_real = np.zeros((len(pm_meta), CFG.N_CLASSES), dtype=np.float32)
matched = 0
for i, rid in enumerate(pm_meta["row_id"].values):
    if rid in row_to_labels:
        for idx in row_to_labels[rid]:
            y_real[i, idx] = 1.0
        matched += 1
print(f"\n✓ Real labels matched: {matched} / {len(pm_meta)} rows")
print(f"  Avg positives per row: {y_real.sum(axis=1).mean():.2f}")
print(f"  Rows with at least 1 positive: {(y_real.sum(axis=1) > 0).sum()}")

In [ ]:
# =============================================================================
# CELL 5 (REPLACEMENT): Load unlabeled data + generate pseudo-labels (v02)
# =============================================================================
# Changes vs v01:
#   • Per-class thresholds at the 97th percentile of each class's logit
#     distribution (instead of single global threshold of 2.0)
#   • Top-K cap per window (max 5 species marked positive per window)
#   • Exclude classes Perch can't see — for those, pseudo-label = 0 everywhere
#   • Verbose stats so we can sanity-check the result before training
# =============================================================================

print("\n" + "=" * 70)
print("Loading unlabeled data + generating pseudo-labels (v02 logic)")
print("=" * 70)

unl_npz = np.load(f"{CFG.EMBED_DATASET}/perch_train_soundscapes_all.npz")
unl_meta = pd.read_parquet(f"{CFG.EMBED_DATASET}/perch_train_soundscapes_meta.parquet")
unl_embs   = unl_npz["embs"]    # (127896, 1536)
unl_scores = unl_npz["scores"]  # (127896, 234) — Perch logits

print(f"Unlabeled embs:    {unl_embs.shape}")
print(f"Unlabeled scores:  {unl_scores.shape}")
print(f"Unlabeled meta:    {unl_meta.shape}")

# ---------------------------------------------------------------------------
# Step 1: Identify classes Perch can actually see
# A class with no Perch signal will have nearly all logits near zero or
# slightly negative. We detect this by checking the max logit.
# ---------------------------------------------------------------------------
class_max_logit = unl_scores.max(axis=0)
PERCH_SIGNAL_THRESHOLD = 1.0   # if max logit across all 127k windows < 1.0, no signal
has_perch_signal = class_max_logit > PERCH_SIGNAL_THRESHOLD

n_with_signal = int(has_perch_signal.sum())
n_no_signal = int((~has_perch_signal).sum())
print(f"\nClasses with Perch signal (max logit > {PERCH_SIGNAL_THRESHOLD}): "
      f"{n_with_signal} / {CFG.N_CLASSES}")
print(f"Classes without Perch signal: {n_no_signal} "
      f"(pseudo-labels for these set to 0 — head can only learn them from real labels)")

# ---------------------------------------------------------------------------
# Step 2: Per-class thresholds at the 97th percentile of each class's logits
# This calibrates to a ~3% positive rate per class, regardless of class's
# overall logit distribution.
# ---------------------------------------------------------------------------
PERCENTILE = 97.0
class_thresholds = np.percentile(unl_scores, PERCENTILE, axis=0).astype(np.float32)

# Floor the threshold at a minimum sensible value to avoid noise positives
# in classes where the 97th percentile is still near zero
MIN_THRESHOLD = 1.5
class_thresholds = np.maximum(class_thresholds, MIN_THRESHOLD)

# For classes without Perch signal, set threshold to +inf (no pseudo-positives)
class_thresholds[~has_perch_signal] = np.inf

print(f"\nPer-class thresholds (percentile = {PERCENTILE}, min floor = {MIN_THRESHOLD}):")
finite_thresh = class_thresholds[np.isfinite(class_thresholds)]
print(f"  min: {finite_thresh.min():.2f}, median: {np.median(finite_thresh):.2f}, "
      f"max: {finite_thresh.max():.2f}")

# ---------------------------------------------------------------------------
# Step 3: Generate raw pseudo-labels using per-class thresholds
# ---------------------------------------------------------------------------
y_pseudo_raw = (unl_scores > class_thresholds[np.newaxis, :]).astype(np.float32)
positives_per_row_raw = y_pseudo_raw.sum(axis=1)
print(f"\nAfter per-class thresholds (before top-K cap):")
print(f"  Average positives per row: {positives_per_row_raw.mean():.2f}")
print(f"  Median:                   {np.median(positives_per_row_raw):.0f}")
print(f"  Max:                      {int(positives_per_row_raw.max())}")
print(f"  Rows with 0 positives:    {int((positives_per_row_raw == 0).sum())}")

# ---------------------------------------------------------------------------
# Step 4: Apply top-K cap per window
# For each row, keep only the K highest-scoring positives.
# ---------------------------------------------------------------------------
TOP_K = 5
y_pseudo = np.zeros_like(y_pseudo_raw)

# Vectorized top-K cap: for each row, find top-K classes among those above threshold
for i in range(len(y_pseudo_raw)):
    pos_idx = np.where(y_pseudo_raw[i] > 0)[0]
    if len(pos_idx) == 0:
        continue
    if len(pos_idx) <= TOP_K:
        y_pseudo[i, pos_idx] = 1.0
    else:
        # Keep top-K by score among the positives
        scores_at_pos = unl_scores[i, pos_idx]
        topk_local = np.argsort(scores_at_pos)[-TOP_K:]
        topk_global = pos_idx[topk_local]
        y_pseudo[i, topk_global] = 1.0

positives_per_row = y_pseudo.sum(axis=1)
print(f"\nAfter top-K cap (K={TOP_K}):")
print(f"  Average positives per row: {positives_per_row.mean():.2f}")
print(f"  Median:                   {np.median(positives_per_row):.0f}")
print(f"  Max:                      {int(positives_per_row.max())}")
print(f"  Rows with 0 positives:    {int((positives_per_row == 0).sum())}")
print(f"  Rows with 1 positive:     {int((positives_per_row == 1).sum())}")
print(f"  Rows with 2-5 positives:  {int(((positives_per_row >= 2) & (positives_per_row <= 5)).sum())}")

# ---------------------------------------------------------------------------
# Step 5: Filter out rows with zero pseudo-positives
# (those windows give no training signal)
# ---------------------------------------------------------------------------
keep_mask = positives_per_row >= CFG.MIN_POSITIVES_PER_PSEUDO
print(f"\nFiltering: keeping {int(keep_mask.sum())} / {len(y_pseudo)} rows "
      f"(dropped {int((~keep_mask).sum())} rows with no pseudo-positives)")
unl_embs_kept   = unl_embs[keep_mask]
y_pseudo_kept   = y_pseudo[keep_mask]
unl_meta_kept   = unl_meta[keep_mask].reset_index(drop=True)
unl_scores_kept = unl_scores[keep_mask]  # save for diagnostic later

# ---------------------------------------------------------------------------
# Step 6: Per-class pseudo-label coverage
# ---------------------------------------------------------------------------
class_counts_pseudo = y_pseudo_kept.sum(axis=0)
print(f"\nPer-class pseudo-label coverage:")
print(f"  min: {int(class_counts_pseudo.min())}, "
      f"median: {int(np.median(class_counts_pseudo))}, "
      f"max: {int(class_counts_pseudo.max())}")
print(f"  Classes with 0 pseudo-positives: {int((class_counts_pseudo == 0).sum())} "
      f"(these can only be learned from the 708 real-labeled windows)")
print(f"  Classes with < 100 pseudo-positives: "
      f"{int((class_counts_pseudo < 100).sum())}")

# Save for diagnostic in Cell 12
PSEUDO_LABEL_CFG_USED = {
    "percentile": PERCENTILE,
    "min_threshold": MIN_THRESHOLD,
    "top_k": TOP_K,
    "perch_signal_threshold": PERCH_SIGNAL_THRESHOLD,
}
print(f"\n✓ Pseudo-label config: {PSEUDO_LABEL_CFG_USED}")

In [ ]:
# =============================================================================
# CELL 6: Build combined training set + GroupKFold split
# =============================================================================
print("\n" + "=" * 70)
print("Building combined train/val splits with GroupKFold by filename")
print("=" * 70)

# Real-label rows (708 windows, treat each row independently)
real_X = pm_embs
real_y = y_real
real_groups = pm_meta["filename"].values  # group by filename for split
real_weights = np.full(len(real_X), CFG.REAL_SAMPLE_WEIGHT, dtype=np.float32)
real_source = np.full(len(real_X), "real", dtype=object)

# Pseudo-label rows (the filtered unlabeled set)
pseudo_X = unl_embs_kept
pseudo_y = y_pseudo_kept
pseudo_groups = unl_meta_kept["filename"].values
pseudo_weights = np.full(len(pseudo_X), CFG.PSEUDO_SAMPLE_WEIGHT, dtype=np.float32)
pseudo_source = np.full(len(pseudo_X), "pseudo", dtype=object)

# Combine
all_X = np.vstack([real_X, pseudo_X])
all_y = np.vstack([real_y, pseudo_y])
all_groups = np.concatenate([real_groups, pseudo_groups])
all_weights = np.concatenate([real_weights, pseudo_weights])
all_source = np.concatenate([real_source, pseudo_source])
print(f"Combined dataset: {len(all_X)} rows ({len(real_X)} real + {len(pseudo_X)} pseudo)")
print(f"Unique groups (files): {len(set(all_groups))}")

# GroupKFold split — IMPORTANT: validate only on REAL labels (pseudo labels are noisy)
# Strategy: split groups, then for validation keep only rows where source == 'real'
# This way training uses both, but validation reflects real-label performance.
gkf = GroupKFold(n_splits=CFG.N_FOLDS)
fold_assign = np.zeros(len(all_X), dtype=int)
for fold, (_, val_idx) in enumerate(gkf.split(all_X, groups=all_groups)):
    fold_assign[val_idx] = fold
print(f"Fold {CFG.FOLD_TO_TRAIN} distribution:")
print(f"  Train rows: {(fold_assign != CFG.FOLD_TO_TRAIN).sum()}")
print(f"  Val rows total: {(fold_assign == CFG.FOLD_TO_TRAIN).sum()}")
print(f"  Val rows that are REAL-labeled: "
      f"{((fold_assign == CFG.FOLD_TO_TRAIN) & (all_source == 'real')).sum()}")

train_mask = fold_assign != CFG.FOLD_TO_TRAIN
val_mask = (fold_assign == CFG.FOLD_TO_TRAIN) & (all_source == "real")

X_train = all_X[train_mask]; y_train = all_y[train_mask]; w_train = all_weights[train_mask]
X_val   = all_X[val_mask];   y_val   = all_y[val_mask];   w_val   = all_weights[val_mask]
print(f"\nFinal splits:")
print(f"  Train: {X_train.shape[0]} rows")
print(f"  Val (real-only): {X_val.shape[0]} rows")
assert X_val.shape[0] > 0, "No real-labeled rows in val fold — check group split"

In [ ]:
# =============================================================================
# CELL 7: Dataset + DataLoader
# =============================================================================
class EmbDataset(Dataset):
    def __init__(self, X, y, weights):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
        self.w = torch.from_numpy(weights).float()
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i], self.w[i]

train_ds = EmbDataset(X_train, y_train, w_train)
val_ds = EmbDataset(X_val, y_val, w_val)
train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                        num_workers=CFG.NUM_WORKERS, pin_memory=True)
print(f"Train batches/epoch: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

In [ ]:
# =============================================================================
# CELL 8: Model
# =============================================================================
class PerchHead(nn.Module):
    """Small MLP head on top of Perch v2 embeddings."""
    def __init__(self, in_dim=1536, hidden_dim=512, n_classes=234, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes),
        )
    def forward(self, x):
        return self.net(x)

model = PerchHead(in_dim=CFG.EMB_DIM, hidden_dim=CFG.HIDDEN_DIM,
                  n_classes=CFG.N_CLASSES, dropout=CFG.DROPOUT).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params:,} (~{n_params * 4 / 1e6:.1f} MB at float32)")

In [ ]:
# =============================================================================
# CELL 9: Optimizer + losses
# =============================================================================
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=1e-6)


def weighted_bce_loss(logits, targets, weights):
    """Multi-label BCE loss with per-sample weights.
    weights: shape (B,), broadcast across classes."""
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")  # (B, C)
    per_sample = bce.mean(dim=1)  # (B,)
    return (per_sample * weights).sum() / weights.sum().clamp(min=1e-6)

In [ ]:
# =============================================================================
# CELL 10: Smoke test (1 batch)
# =============================================================================
print("=" * 70)
print("SMOKE TEST: 1 train batch + 1 val batch")
print("=" * 70)
model.train()
xb, yb, wb = next(iter(train_loader))
xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)
print(f"X batch: {xb.shape}, y batch: {yb.shape}, weights: {wb.shape}")
print(f"X stats: mean={xb.mean().item():.3f}, std={xb.std().item():.3f}")
print(f"y positives per row: {yb.sum(dim=1).mean().item():.2f}")
print(f"weights range: [{wb.min().item():.2f}, {wb.max().item():.2f}]")

logits = model(xb)
print(f"logits: shape={tuple(logits.shape)}, range=[{logits.min().item():.2f}, {logits.max().item():.2f}]")
loss = weighted_bce_loss(logits, yb, wb)
print(f"loss: {loss.item():.4f}")
assert torch.isfinite(loss), "Non-finite loss on smoke test"
loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
print(f"grad_norm: {grad_norm.item():.4f}")
assert torch.isfinite(grad_norm), "Non-finite grad on smoke test"
optimizer.zero_grad()
print("✓ SMOKE TEST PASSED")

In [ ]:
# =============================================================================
# CELL 11: Train loop
# =============================================================================
def macro_auc_safe(y_true, y_score):
    """ROC-AUC averaged over classes that have at least one positive."""
    aucs = []
    for c in range(y_true.shape[1]):
        if y_true[:, c].sum() == 0:
            continue
        try:
            aucs.append(roc_auc_score(y_true[:, c], y_score[:, c]))
        except Exception:
            continue
    return float(np.mean(aucs)) if aucs else 0.0, len(aucs)


def train_one_epoch():
    model.train()
    losses = []
    for xb, yb, wb in train_loader:
        xb, yb, wb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True), wb.to(device, non_blocking=True)
        logits = model(xb)
        loss = weighted_bce_loss(logits, yb, wb)
        if not torch.isfinite(loss):
            raise RuntimeError("NaN/Inf loss during training")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))


@torch.no_grad()
def validate():
    model.eval()
    all_y, all_p = [], []
    for xb, yb, _ in val_loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        logits = model(xb)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_p.append(probs)
        all_y.append(yb.cpu().numpy())
    y_true = np.concatenate(all_y); y_pred = np.concatenate(all_p)
    y_pred = np.nan_to_num(y_pred, nan=0.5)
    auc, n_scored = macro_auc_safe(y_true, y_pred)
    return auc, n_scored, y_true, y_pred


print("=" * 70)
print("TRAINING")
print("=" * 70)
best_auc = -1.0
best_path = os.path.join(CFG.OUT_DIR, "head_v01.pth")
log_path = os.path.join(CFG.OUT_DIR, "train_log.json")
log = []

for ep in range(CFG.EPOCHS):
    t0 = time.time()
    train_loss = train_one_epoch()
    val_auc, n_scored, y_true_val, y_pred_val = validate()
    scheduler.step()
    dt = time.time() - t0
    print(f"epoch {ep:02d} | train_loss={train_loss:.4f} | "
          f"val_macroAUC={val_auc:.4f} ({n_scored} cls) | {dt:.0f}s")
    log.append({"epoch": ep, "train_loss": train_loss, "val_auc": val_auc,
                "n_scored": n_scored, "time_s": dt})
    with open(log_path, "w") as f:
        json.dump(log, f, indent=2)

    if ep == 0 and val_auc < CFG.SANITY_AUC_FLOOR:
        raise RuntimeError(
            f"Epoch 0 val_auc={val_auc:.4f} < floor {CFG.SANITY_AUC_FLOOR}. "
            f"Head is not learning — check pseudo-label threshold or data alignment."
        )

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({
            "state_dict": model.state_dict(),
            "species": SPECIES,
            "cfg": {k: getattr(CFG, k) for k in dir(CFG) if not k.startswith("_")},
            "epoch": ep, "val_auc": val_auc,
        }, best_path)
        print(f"   ↑ saved best (auc={best_auc:.4f})")

print(f"\nBest val macro-AUC: {best_auc:.4f}")

In [ ]:
# =============================================================================
# CELL 12 (REPLACEMENT): Per-class diagnostic + Option D go/no-go criteria
# =============================================================================
# This cell evaluates whether to proceed to Day 4 integration based on three
# criteria:
#   1. val_macroAUC ≥ 0.80
#   2. ≥ 30 classes with per-class val AUC ≥ 0.85
#   3. Head predictions are meaningfully different from Perch logits
#      (Spearman correlation between 0.7 and 0.9 globally)
# =============================================================================

print("=" * 70)
print("Per-class AUC diagnostic + Option D go/no-go evaluation")
print("=" * 70)

# Reload best checkpoint for diagnostic
ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["state_dict"])
val_auc, n_scored, y_true_val, y_pred_val = validate()

# Per-class AUC
per_class = []
for c in range(CFG.N_CLASSES):
    n_pos = int(y_true_val[:, c].sum())
    if n_pos == 0:
        per_class.append({"species": SPECIES[c], "n_pos": 0, "auc": np.nan})
        continue
    try:
        auc_c = roc_auc_score(y_true_val[:, c], y_pred_val[:, c])
    except Exception:
        auc_c = np.nan
    per_class.append({"species": SPECIES[c], "n_pos": n_pos, "auc": auc_c})

pc = pd.DataFrame(per_class)
scored = pc.dropna(subset=["auc"])

# Taxonomic breakdown
tax = pd.read_csv(f"{CFG.COMP_DIR}/taxonomy.csv")
tax_map = dict(zip(tax["primary_label"].astype(str), tax["class_name"]))
scored = scored.assign(tax_class=scored["species"].map(tax_map))

print(f"\nMean AUC over {len(scored)} scoreable classes: {scored['auc'].mean():.4f}")
print(f"Median AUC: {scored['auc'].median():.4f}")
print(f"\nDistribution:")
for thresh in [0.5, 0.7, 0.8, 0.85, 0.9, 0.95]:
    n = int((scored["auc"] >= thresh).sum())
    print(f"  AUC ≥ {thresh}: {n} / {len(scored)} ({100*n/len(scored):.1f}%)")
print(f"\nBy taxonomic class:")
print(scored.groupby("tax_class")["auc"].agg(["count", "mean", "median"]).round(3))

print(f"\nWorst 10 classes:")
print(scored.nsmallest(10, "auc").to_string(index=False))

scored.to_csv(os.path.join(CFG.OUT_DIR, "per_class_auc.csv"), index=False)


# ---------------------------------------------------------------------------
# Criterion 3 diagnostic: correlate head predictions with Perch logits
# We score the head on a sample of unlabeled windows and compare to the
# raw Perch logits that the head was trained against. High correlation =
# head learned to mimic Perch (bad — no diversity). Moderate correlation =
# head learned something distinct (good).
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("Diversity diagnostic: head vs Perch logits correlation")
print("-" * 70)

# Sample 2000 unlabeled windows for the diversity check
rng = np.random.RandomState(CFG.SEED)
sample_idx = rng.choice(len(unl_embs_kept), size=min(2000, len(unl_embs_kept)), replace=False)
sample_embs   = unl_embs_kept[sample_idx]
sample_perch  = unl_scores_kept[sample_idx]  # Perch logits

# Score these with the head
model.eval()
with torch.no_grad():
    sx = torch.from_numpy(sample_embs).float().to(device)
    head_logits = model(sx).cpu().numpy()

# Spearman correlation (rank-based, robust to scale) per class, then global
from scipy.stats import spearmanr
per_class_corrs = []
for c in range(CFG.N_CLASSES):
    if sample_perch[:, c].std() < 1e-6:
        continue
    try:
        r, _ = spearmanr(sample_perch[:, c], head_logits[:, c])
        if np.isfinite(r):
            per_class_corrs.append(r)
    except Exception:
        continue

if per_class_corrs:
    mean_corr = float(np.mean(per_class_corrs))
    median_corr = float(np.median(per_class_corrs))
    print(f"Spearman corr (head vs Perch logits), per-class:")
    print(f"  mean:   {mean_corr:.3f}")
    print(f"  median: {median_corr:.3f}")
    print(f"  range:  [{min(per_class_corrs):.3f}, {max(per_class_corrs):.3f}]")
else:
    mean_corr = float("nan")
    median_corr = float("nan")
    print("No correlation could be computed.")


# ---------------------------------------------------------------------------
# Go/No-Go evaluation
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("OPTION D GO/NO-GO EVALUATION")
print("=" * 70)

best_val_auc = ckpt.get("val_auc", val_auc)
n_strong_classes = int((scored["auc"] >= 0.85).sum())

c1 = best_val_auc >= 0.80
c2 = n_strong_classes >= 30
c3 = 0.5 <= median_corr <= 0.92 if np.isfinite(median_corr) else False

print(f"\nCriterion 1 — val_macroAUC ≥ 0.80:")
print(f"  Actual: {best_val_auc:.4f}")
print(f"  Result: {'✓ PASS' if c1 else '✗ FAIL'}")

print(f"\nCriterion 2 — ≥ 30 classes with AUC ≥ 0.85:")
print(f"  Actual: {n_strong_classes} classes")
print(f"  Result: {'✓ PASS' if c2 else '✗ FAIL'}")

print(f"\nCriterion 3 — head/Perch median correlation in [0.5, 0.92]:")
print(f"  Actual: {median_corr:.3f}")
print(f"  Result: {'✓ PASS' if c3 else '✗ FAIL'}")

n_passed = int(c1) + int(c2) + int(c3)
print(f"\n{'='*70}")
print(f"VERDICT: {n_passed} / 3 criteria passed")

if n_passed == 3:
    print("→ PROCEED to Day 4 integration")
elif n_passed == 2:
    print("→ MIXED — consider Option B for richer pseudo-labels before Day 4")
else:
    print("→ HEAD NOT VIABLE — recommend Option C (lock exp019, focus on paper)")

print(f"\nPaste this output to Claude for next-step decision.")

In [ ]:
# =============================================================================
# CELL 13: Done
# =============================================================================
print("=" * 70)
print("DONE")
print("=" * 70)
print(f"""
Files saved to /kaggle/working/:
  - head_v01.pth     (best checkpoint)
  - train_log.json   (per-epoch metrics)
  - per_class_auc.csv (per-class validation AUC)

Next steps:
1. Commit-Run this notebook (Save & Run All Commit) so outputs land in the
   Output tab and can be saved as a Kaggle Dataset.
2. From the committed version's Output tab, create a Kaggle Dataset
   containing head_v01.pth (and optionally the log/diagnostic files).
3. Name: birdclef-perch-head-v01
4. This checkpoint will be loaded by the Day 4 integration notebook
   into the exp019 pipeline as an additional ensemble member.

Best val macro-AUC (real labels only): {best_auc:.4f}
""")